# Live slate

Set `TEAM` to an abbreviation on the latest Pinnacle board. Odds come from the newest `pinnacle_nba_*_team.json`. Starters come from the newest Rotowire file that day; a team that has already left the page keeps the last five posted earlier that day.

Layer 1 draws one game environment for that team. Layer 2 is the pregame role for each player on that team and on the opponent: his minutes, his team's form, and the opponent's form. This notebook is the live path, so those windows include every completed game before the slate. The Layer 1 draw is the script. It is not drawn again.

Layer 3 turns that role and that script into minutes. The live minutes are rotation segments. A segment is five players on the floor for a positive duration. Posted start, who is available, and the blowout flag on that row's margin choose the fives. The durations add up to the script's game minutes, so the team total is five times that length and no player plays longer than that length. A player Rotowire lists as out, or out for the season, is unavailable and gets 0 minutes. Questionable and doubtful players stay available. The conditional fit still trains on every completed game before the slate, with each window stopping before the game being scored. That fit is the `state` path. These live minutes are the segments, so they are not rescaled through it.

Layer 4 turns those minutes and the possession counts into shots and turnovers. The attempt pool follows the team's possessions. Each player's share follows his minutes and his recent attempts per minute. The shot profile uses the same windows: training stops before the game being scored, and the live profile keeps the latest completed game. Makes and free throws then move so the players' points on that row sum to the nearest whole number of that side's scenario points.

Layer 5 credits the assists, rebounds, steals, and blocks those shots and turnovers already produced. A make can be assisted by a teammate. A miss is split between the two teams, then among the teammates who were on the floor. A steal comes from the opponent's turnovers, and a block comes from an opponent miss. The same windows decide who is credited.

Layer 6 keeps one simulation row as one box score. A prop clears when the stat is strictly above its line. The chance that two props both clear is the share of rows where both do, which is not the product of the two separate chances.

In [1]:
import argparse
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

def repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "nba_game_simulator").is_dir() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError("open this notebook from the nba_quant_analysis repo")

REPO = repo_root()
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "nba_game_simulator" / "src"))

from game_enviroment import (
    build_features,
    build_game_table,
    load_gamelogs,
    load_latest_slate,
    load_model,
    print_sim_summary,
    save_model,
    season_of,
    simulate_game,
    train,
)
from lineup import (
    attach_lineup_fill,
    build_lineup_features,
    fill_lineup_gaps,
    merge_lineup_features,
)
from role_context import attach_script, role_for_slate
from box_score import Prop, assemble_box_score, prop_probability
from shared_events import allocate_events, fit_events, live_events, training_events
from possession_allocation import (
    allocate_opportunities,
    fit_opportunities,
    live_opportunities,
    training_opportunities,
)
from rotations import (
    allocate_rotation,
    build_rotation_segments,
    fit_rotation,
    invalid_segment_count,
    training_minutes,
)
from src.live.predict_points import lineups_on
from src.pipeline.silver.rotowire import normalize_team_abbreviation
from src.scrapers.arbitrage import canonical_team

TEAM = "LAL"
MODEL_PATH = REPO / "nba_game_simulator" / "game_env_model.json"

In [2]:
log_paths = sorted((REPO / "data" / "silver" / "nba").glob("*/regular_season/player_gamelogs.parquet"))
if not log_paths:
    raise FileNotFoundError("no player gamelogs under data/silver/nba")
roster_path = log_paths[-1]
played = pd.read_parquet(
    roster_path,
    columns=["player_name", "player_id", "team_abbreviation", "game_date"],
)
played["game_date"] = pd.to_datetime(played["game_date"])
roster = (
    played.sort_values(["player_id", "game_date"])
    .groupby("player_id", as_index=False)
    .tail(1)[["player_name", "player_id", "team_abbreviation"]]
)

slate = load_latest_slate(TEAM, roster)
board = json.loads(Path(slate["pinnacle_file"]).read_text())["games"]
posted = lineups_on(slate["date"])

def abbr_for(label):
    wanted = canonical_team(label, "nba")
    for abbr in roster["team_abbreviation"].dropna().unique():
        normalized = normalize_team_abbreviation(abbr)
        if canonical_team(normalized, "nba") == wanted:
            return normalized
    return label

rows = []
for game in board:
    participants = game.get("participants") or []
    if len(participants) < 2:
        continue
    away_name, home_name = participants[0], participants[1]
    markets = game.get("team_markets") or {}
    total = None
    for block in markets.get("total") or []:
        if block.get("period") not in (0, None) or block.get("is_alternate"):
            continue
        for line in block.get("lines") or []:
            if line.get("side") == "over" and line.get("points") is not None:
                total = line["points"]
                break
    home_spread = None
    for block in markets.get("spread") or []:
        if block.get("period") not in (0, None) or block.get("is_alternate"):
            continue
        for line in block.get("lines") or []:
            if line.get("side") == "home" and line.get("points") is not None:
                home_spread = line["points"]
                break
    home_abbr = abbr_for(home_name)
    away_abbr = abbr_for(away_name)
    rows.append(
        {
            "away": away_abbr,
            "home": home_abbr,
            "home_spread": home_spread,
            "total": total,
            "away_five": " ".join(posted.get(away_abbr) or []) or "—",
            "home_five": " ".join(posted.get(home_abbr) or []) or "—",
        }
    )

print(f"Pinnacle: {Path(slate['pinnacle_file']).name}")
print(f"Rotowire: {Path(slate['lineup_file']).name}")
print(f"roster:   {roster_path.relative_to(REPO)} through {played['game_date'].max().date()}")
display(pd.DataFrame(rows))

print(
    f"{slate['team']} vs {slate['opponent']}"
    f"  home={slate['is_home']}  spread={slate['spread']}  total={slate['total']}"
)
if slate["team_lineup"] is None or slate["opponent_lineup"] is None:
    print("A five is missing from today's Rotowire files. That side uses the training-average starters.")
else:
    print(f"starter ids  {slate['team']}: {slate['team_lineup']}")
    print(f"starter ids  {slate['opponent']}: {slate['opponent_lineup']}")
for side, missing in (
    (slate["team"], slate["team_out_absent"]),
    (slate["opponent"], slate["opponent_out_absent"]),
):
    if missing:
        print(
            f"{side} listed out with no game on this roster, so no minutes:"
            f" {', '.join(missing)}"
        )

Pinnacle: pinnacle_nba_2026-11-06_010000_team.json
Rotowire: rotowire_nba_2026-11-06_010000.json
roster:   data/silver/nba/2026-27/regular_season/player_gamelogs.parquet through 2026-11-04


,away,home,home_spread,total,away_five,home_five
0,BOS,CHO,8.5,210.5,Derrick White Hugo Gonzalez Paul George Jayson...,Coby White Sion James Brandon Miller Naz Reid ...
1,MIN,NOP,4.5,230.5,LaMelo Ball Ayo Dosunmu Jonathan Kuminga Jaden...,Dejounte Murray Jeremiah Fears Trey Murphy Zio...
2,OKC,MEM,3.5,225.5,Shai Gilgeous-Alexander Ajay Mitchell Kenrich ...,Ty Jerome Cedric Coward Jaylen Wells Cameron B...
3,PHI,CLE,2.5,228.5,Tyrese Maxey VJ Edgecombe Jaylen Brown LeBron ...,James Harden Donovan Mitchell Peyton Watson Ev...
4,POR,LAL,-2.5,233.0,Damian Lillard Scoot Henderson Toumani Camara ...,Luka Doncic Austin Reaves Ziaire Williams Mati...
5,SAS,SAC,15.0,229.5,De'Aaron Fox Stephon Castle Dylan Harper Devin...,Darius Acuff Zach LaVine De'Andre Hunter Preci...
6,TOR,MIL,7.0,231.0,Immanuel Quickley RJ Barrett Ja'Kobe Walter Sc...,Brayden Burries Tyler Herro Jaime Jaquez Kyle ...
7,UTA,DAL,-7.5,226.5,Keyonte George Darryn Peterson Ace Bailey Laur...,Kyrie Irving Max Christie Cooper Flagg P.J. Wa...
8,WAS,ATL,-6.5,229.5,Trae Young Kyshawn George AJ Dybantsa Anthony ...,Kingston Flemings Nickeil Alexander-Walker Dys...


LAL vs POR  home=True  spread=-2.5  total=233.0
starter ids  LAL: [1629029, 1630559, 1630533, 1629680, 1631117]
starter ids  POR: [203081, 1630703, 1641739, 1630166, 1642270]
POR listed out with no game on this roster, so no minutes: Shaedon Sharpe


In [3]:
if MODEL_PATH.exists():
    model = load_model(MODEL_PATH)
    print(f"loaded {MODEL_PATH.name}  ({model.get('trained_games')} games)")
else:
    raw = load_gamelogs(str(REPO / "data" / "silver" / "nba" / "*" / "regular_season" / "player_gamelogs.parquet"))
    games = build_game_table(raw)
    lineups, lineup_state = build_lineup_features(
        raw, halflife=12.0, prior_weight=4.0, season_carry=0.6, window=300,
    )
    games = merge_lineup_features(games, lineups)
    lineup_state = attach_lineup_fill(lineup_state, games)
    games = fill_lineup_gaps(games)
    games, state = build_features(
        games, halflife=12.0, prior_w=4.0, carry=0.6, window=300, min_league=30,
    )
    state["lineup"] = lineup_state
    args = argparse.Namespace(
        test_seasons=1,
        halflife=12.0,
        prior_weight=4.0,
        season_carry=0.6,
        league_window=300,
        plots=False,
    )
    model = train(games, state, args)
    save_model(model, str(MODEL_PATH))

sim = simulate_game(
    model,
    slate["team"],
    slate["opponent"],
    slate["date"],
    total=slate["total"],
    spread=slate["spread"],
    is_home=slate["is_home"],
    team_lineup=slate["team_lineup"],
    opponent_lineup=slate["opponent_lineup"],
    n=5000,
    seed=7,
)
print_sim_summary(sim, slate["total"], slate["spread"])
sim[["team_poss", "opp_poss", "pace", "team_pts", "opp_pts", "game_total", "team_margin", "n_ot"]].describe().round(2)

loaded game_env_model.json  (8365 games)

Game environment: POR at LAL  | total 233.0 | team spread -2.5  | rest team/opponent 2/2 days
               mean     sd    p05     p25     p50     p75     p95
pace          99.87   3.78  93.66   97.42   99.87  102.37  106.15
team_poss    100.50   4.80  93.62   97.53  100.10  102.89  108.67
team_pts     117.22  11.37  98.47  109.68  117.48  124.94  135.24
opp_pts      115.25  11.42  96.33  107.60  115.15  122.83  134.14
team_margin    1.97  15.67 -24.18   -8.23    2.00   12.64   26.90
  P(OT) = 5.3%


,team_poss,opp_poss,pace,team_pts,opp_pts,game_total,team_margin,n_ot
count,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00
mean,100.50,100.49,99.87,117.22,115.25,232.48,1.97,0.06
std,4.80,4.81,3.78,11.37,11.42,16.54,15.67,0.27
min,86.99,87.28,87.34,77.78,67.79,167.82,-57.26,0.00
25%,97.53,97.51,97.42,109.68,107.60,221.09,-8.23,0.00
50%,100.10,100.12,99.87,117.48,115.15,232.54,2.00,0.00
75%,102.89,102.88,102.37,124.94,122.83,243.53,12.64,0.00
max,136.53,136.73,113.49,155.49,155.32,306.63,62.99,3.00


In [4]:
ROLE_COLUMNS = [
    "season_year",
    "player_id",
    "player_name",
    "team_id",
    "team_abbreviation",
    "game_id",
    "game_date",
    "matchup",
    "min",
    "pf",
    "team_pace",
    "team_off_rating",
    "team_def_rating",
    "pace",
    "off_rating",
    "def_rating",
    "start_position",
    "team_pts",
    "opp_pts",
    "player_team_spread",
    "fga",
    "fgm",
    "fg3_a",
    "fg3_m",
    "fta",
    "ftm",
    "tov",
    "team_fga",
    "team_poss",
    "team_tov",
    "ast",
    "oreb",
    "dreb",
    "stl",
    "blk",
    "team_fgm",
    "team_ast",
    "team_oreb",
    "team_stl",
    "team_blk",
    "opp_dreb",
    "opp_tov",
    "opp_fga",
]
logs = pd.concat(
    [pd.read_parquet(path, columns=ROLE_COLUMNS) for path in log_paths],
    ignore_index=True,
)
logs["game_date"] = pd.to_datetime(logs["game_date"])
slate_day = pd.Timestamp(slate["date"])
history = logs.loc[logs["game_date"] < slate_day]
if history.empty:
    raise ValueError(f"no completed games before {slate['date']}")
# Who is on the team comes from this season. The windows below still use earlier seasons.
this_season = history.loc[history["season_year"].astype(str).eq(season_of(slate_day))]
latest = (
    this_season.sort_values(["player_id", "game_date", "game_id"])
    .groupby("player_id", as_index=False)
    .tail(1)
)
latest["abbr"] = latest["team_abbreviation"].map(normalize_team_abbreviation)


def team_id_for(abbr: str) -> int:
    ids = latest.loc[latest["abbr"].eq(abbr), "team_id"].dropna()
    if ids.empty:
        raise ValueError(f"no completed games for {abbr} before {slate['date']}")
    return int(ids.mode().iloc[0])


def players_for(abbr, own_id, other_id, is_home, spread, starter_ids, out_ids):
    """Players last seen on this team, plus anyone in the posted five.

    ``out_ids`` are Rotowire out and out-for-season players. They stay on the
    roster with ``available`` 0, so the segment schedule gives them 0 minutes.
    """
    starters = {int(player_id) for player_id in (starter_ids or [])}
    outs = {int(player_id) for player_id in (out_ids or [])}
    side = latest.loc[latest["abbr"].eq(abbr)]
    posted = latest.loc[latest["player_id"].astype(int).isin(starters | outs)]
    missing = (starters | outs).difference(posted["player_id"].astype(int))
    if missing:
        print(f"{abbr} posted players missing from logs: {sorted(missing)}")
    side = pd.concat([side, posted], ignore_index=True).drop_duplicates("player_id")
    if side.empty:
        raise ValueError(f"no players for {abbr} before {slate['date']}")
    on_roster = side["player_id"].astype(int)
    return pd.DataFrame(
        {
            "player_id": side["player_id"].to_numpy(),
            "player_name": side["player_name"].to_numpy(),
            "team": abbr,
            "game_date": slate_day,
            "team_id": own_id,
            "opp_team_id": other_id,
            "is_home": float(bool(is_home)),
            "is_starter": on_roster.isin(starters).astype(float).to_numpy(),
            "available": (~on_roster.isin(outs)).astype(float).to_numpy(),
            "team_spread": float(spread),
        }
    )


team_abbr = normalize_team_abbreviation(slate["team"])
opp_abbr = normalize_team_abbreviation(slate["opponent"])
team_id = team_id_for(team_abbr)
opp_id = team_id_for(opp_abbr)
players = pd.concat(
    [
        players_for(
            team_abbr,
            team_id,
            opp_id,
            slate["is_home"],
            slate["spread"],
            slate["team_lineup"],
            slate["team_out"],
        ),
        players_for(
            opp_abbr,
            opp_id,
            team_id,
            not slate["is_home"],
            -float(slate["spread"]),
            slate["opponent_lineup"],
            slate["opponent_out"],
        ),
    ],
    ignore_index=True,
)
names = players[["player_id", "player_name", "team", "available"]]
role = role_for_slate(logs, players.drop(columns=["player_name", "team"]))
role = names.merge(role, on="player_id", how="left", validate="one_to_one")
role["side"] = role["team"].eq(team_abbr).map({True: 0, False: 1})
role = role.sort_values(
    ["side", "is_starter", "min_mean_10"],
    ascending=[True, False, False],
).drop(columns="side")

shown = [
    "team",
    "player_name",
    "is_starter",
    "available",
    "min_mean_10",
    "min_std_10",
    "start_rate_10",
    "pf_mean_10",
    "season_games_prior",
    "team_pace_10",
    "team_off_10",
    "team_def_10",
    "team_rest",
    "team_b2b",
    "opp_pace_10",
    "opp_off_10",
    "opp_def_10",
    "opp_rest",
    "opp_b2b",
    "team_spread",
]
print(
    f"live role through {history['game_date'].max().date()} for {slate_day.date()}"
    f"  ({len(role)} players, latest game included)"
)
out_now = role.loc[role["available"].eq(0), ["team", "player_name"]]
if out_now.empty:
    print("Rotowire out: none on these rosters")
else:
    listed = ", ".join(
        f"{row.team} {row.player_name}" for row in out_now.itertuples(index=False)
    )
    print(f"Rotowire out: {listed}")
display(role[shown].round(2))

team_role = role.loc[role["team"].eq(team_abbr)].reset_index(drop=True)
opp_role = role.loc[role["team"].eq(opp_abbr)].reset_index(drop=True)
opp_env = sim.copy()
opp_env["team_margin"] = -opp_env["team_margin"]
attached = {
    team_abbr: attach_script(team_role, sim),
    opp_abbr: attach_script(opp_role, opp_env),
}
for label, frame in attached.items():
    draws = frame.drop_duplicates("sim_id")
    one = frame.loc[frame["sim_id"].eq(0), "team_final_margin"]
    if one.nunique() != 1:
        raise AssertionError(f"{label} players do not share one margin")
    print(
        f"{label}: {frame['player_id'].nunique()} players share the draw"
        f"  mean margin {draws['team_final_margin'].mean():.1f}"
        f"  P(OT) {draws['went_ot'].mean():.1%}"
        f"  P(blowout) {draws['blowout'].mean():.1%}"
    )


live role through 2026-11-04 for 2026-11-06  (32 players, latest game included)
Rotowire out: POR Ja Morant, POR Vít Krejčí


,team,player_name,is_starter,available,min_mean_10,min_std_10,start_rate_10,pf_mean_10,season_games_prior,team_pace_10,team_off_10,team_def_10,team_rest,team_b2b,opp_pace_10,opp_off_10,opp_def_10,opp_rest,opp_b2b,team_spread
7,LAL,Austin Reaves,1.0,1.0,35.01,4.12,1.00,2.70,9.0,101.20,121.14,111.66,2.0,0.0,100.35,118.54,109.15,2.0,0.0,-2.5
3,LAL,Luka Dončić,1.0,1.0,34.50,3.92,1.00,2.40,8.0,101.20,121.14,111.66,2.0,0.0,100.35,118.54,109.15,2.0,0.0,-2.5
10,LAL,Walker Kessler,1.0,1.0,30.36,2.51,1.00,2.90,9.0,101.20,121.14,111.66,2.0,0.0,100.35,118.54,109.15,2.0,0.0,-2.5
6,LAL,Ziaire Williams,1.0,1.0,24.59,4.58,0.70,2.70,6.0,101.20,121.14,111.66,2.0,0.0,100.35,118.54,109.15,2.0,0.0,-2.5
5,LAL,Matisse Thybulle,1.0,1.0,22.57,6.12,0.90,2.00,9.0,101.20,121.14,111.66,2.0,0.0,100.35,118.54,109.15,2.0,0.0,-2.5
4,LAL,Quentin Grimes,0.0,1.0,26.28,5.52,0.10,1.90,9.0,101.20,121.14,111.66,2.0,0.0,100.35,118.54,109.15,2.0,0.0,-2.5
11,LAL,Jake LaRavia,0.0,1.0,24.23,5.98,0.60,3.00,8.0,101.20,121.14,111.66,2.0,0.0,100.35,118.54,109.15,2.0,0.0,-2.5
8,LAL,Sandro Mamukelashvili,0.0,1.0,22.83,3.93,0.00,1.70,9.0,101.20,121.14,111.66,2.0,0.0,100.35,118.54,109.15,2.0,0.0,-2.5
1,LAL,Collin Sexton,0.0,1.0,19.35,4.90,0.20,2.60,8.0,101.20,121.14,111.66,2.0,0.0,100.35,118.54,109.15,2.0,0.0,-2.5
2,LAL,Jarred Vanderbilt,0.0,1.0,13.63,6.79,0.00,1.40,5.0,101.20,121.14,111.66,2.0,0.0,100.35,118.54,109.15,2.0,0.0,-2.5


LAL: 16 players share the draw  mean margin 2.0  P(OT) 5.3%  P(blowout) 20.3%
POR: 16 players share the draw  mean margin -2.0  P(OT) 5.3%  P(blowout) 20.3%


In [5]:
trained = training_minutes(history)
rotation = fit_rotation(trained)
print(
    f"rotation fit on {len(trained):,} player-games"
    f" through {history['game_date'].max().date()}"
    "  (each window stops before that game)"
)

rng = np.random.default_rng(7)
minutes = {
    team_abbr: allocate_rotation(
        rotation, team_role, sim, rng, method="segments"
    ),
    opp_abbr: allocate_rotation(
        rotation, opp_role, opp_env, rng, method="segments"
    ),
}


def script_zero(label, side, environment, drawn):
    """One attached script. Segments for that row, then the minute row."""
    row = environment.iloc[0]
    game_minutes = float(row["game_minutes"])
    n_ot = float(row["n_ot"])
    margin = float(row["team_margin"])
    segments = build_rotation_segments(
        side,
        game_minutes=game_minutes,
        n_ot=n_ot,
        team_margin=margin,
    )
    invalid = invalid_segment_count(
        segments,
        side,
        game_minutes=game_minutes,
        n_ot=n_ot,
    )
    if invalid != 0:
        raise AssertionError(f"{label} script 0 has {invalid} invalid segments")
    total = float(drawn[0].sum())
    if abs(total - 5.0 * game_minutes) > 1e-4:
        raise AssertionError(f"{label} script 0 misses the team minute budget")
    print(
        f"{label} script 0: n_ot {int(round(n_ot))}  "
        f"team_margin {margin:+.1f}  "
        f"cap {game_minutes:.1f}  "
        f"team total {total:.1f}  "
        f"invalid segments {invalid}"
    )


script_zero(team_abbr, team_role, sim, minutes[team_abbr])
script_zero(opp_abbr, opp_role, opp_env, minutes[opp_abbr])


def minute_table(label, side, drawn):
    budget = 5.0 * sim["game_minutes"].to_numpy()
    gap = np.max(np.abs(drawn.sum(axis=1) - budget))
    if gap > 1e-4:
        raise AssertionError(f"{label} minutes miss the script budget by {gap:.4f}")
    cap = sim["game_minutes"].to_numpy()[:, None]
    if np.any(drawn > cap + 1e-6):
        raise AssertionError(f"{label} has a player above the game length")
    summary = pd.DataFrame(
        {
            "team": label,
            "player_name": side["player_name"].to_numpy(),
            "is_starter": side["is_starter"].to_numpy(),
            "min_mean_10": side["min_mean_10"].to_numpy(),
            "mean": drawn.mean(axis=0),
            "sd": drawn.std(axis=0),
            "p10": np.percentile(drawn, 10, axis=0),
            "p90": np.percentile(drawn, 90, axis=0),
        }
    )
    return summary.sort_values(["is_starter", "mean"], ascending=[False, False])


shown_minutes = pd.concat(
    [
        minute_table(team_abbr, team_role, minutes[team_abbr]),
        minute_table(opp_abbr, opp_role, minutes[opp_abbr]),
    ],
    ignore_index=True,
)
team_means = shown_minutes.groupby("team")["mean"].sum().round(1)
print(
    f"segment minutes for {slate_day.date()}  "
    f"mean team total {team_means.to_dict()}"
)
print("sd is the spread across scripts; one script has a fixed schedule")
display(shown_minutes.round(1))


rotation fit on 179,467 player-games through 2026-11-04  (each window stops before that game)
LAL script 0: n_ot 0  team_margin -8.1  cap 48.0  team total 240.0  invalid segments 0
POR script 0: n_ot 0  team_margin +8.1  cap 48.0  team total 240.0  invalid segments 0
segment minutes for 2026-11-06  mean team total {'LAL': 241.5, 'POR': 241.5}
sd is the spread across scripts; one script has a fixed schedule


,team,player_name,is_starter,min_mean_10,mean,sd,p10,p90
0,LAL,Luka Dončić,1.0,34.5,40.5,4.0,33.0,42.0
1,LAL,Walker Kessler,1.0,30.4,40.5,4.0,33.0,42.0
2,LAL,Austin Reaves,1.0,35.0,34.5,4.0,27.0,36.0
3,LAL,Ziaire Williams,1.0,24.6,29.1,2.9,24.0,30.0
4,LAL,Matisse Thybulle,1.0,22.6,29.1,2.9,24.0,30.0
5,LAL,Quentin Grimes,0.0,26.3,30.0,0.3,30.0,30.0
6,LAL,Jake LaRavia,0.0,24.2,24.6,1.3,24.0,27.0
7,LAL,Sandro Mamukelashvili,0.0,22.8,7.8,3.6,6.0,15.0
8,LAL,Collin Sexton,0.0,19.3,2.4,4.9,0.0,12.0
9,LAL,Jarred Vanderbilt,0.0,13.6,2.4,4.9,0.0,12.0


In [6]:
trained_shots = training_opportunities(history)
shot_model = fit_opportunities(trained_shots)
shot_profile = live_opportunities(
    history, players.drop(columns=["player_name", "team"])
)
shot_profile = names.merge(
    shot_profile, on="player_id", how="left", validate="one_to_one"
)
print(
    f"shot profile fit on {len(trained_shots):,} player-games"
    f" through {history['game_date'].max().date()}"
    "  (each window stops before that game)"
)


def side_profile(side):
    ordered = (
        shot_profile.set_index("player_id").loc[side["player_id"]].reset_index()
    )
    if len(ordered) != len(side) or not ordered["player_id"].equals(side["player_id"]):
        raise AssertionError("shot profile does not match the rotation")
    return ordered


shots = {
    team_abbr: allocate_opportunities(
        shot_model,
        side_profile(team_role),
        minutes[team_abbr],
        sim["team_poss"].to_numpy(dtype=float),
        rng,
        team_points=sim["team_pts"].to_numpy(dtype=float),
    ),
    opp_abbr: allocate_opportunities(
        shot_model,
        side_profile(opp_role),
        minutes[opp_abbr],
        sim["opp_poss"].to_numpy(dtype=float),
        rng,
        team_points=sim["opp_pts"].to_numpy(dtype=float),
    ),
}
for label, scenario in ((team_abbr, sim["team_pts"]), (opp_abbr, sim["opp_pts"])):
    target = np.floor(scenario.to_numpy(dtype=float) + 0.5).astype(np.int64)
    if not np.array_equal(shots[label]["PTS"].sum(axis=1), target):
        raise AssertionError(f"{label} points miss the scenario score")


def shot_table(label, side, drawn):
    if not np.array_equal(drawn["FGA"].sum(axis=1), drawn["team_fga"]):
        raise AssertionError(f"{label} attempts do not sum to the shot pool")
    if not np.array_equal(drawn["TOV"].sum(axis=1), drawn["team_tov"]):
        raise AssertionError(f"{label} turnovers do not sum to the turnover pool")
    summary = pd.DataFrame(
        {
            "team": label,
            "player_name": side["player_name"].to_numpy(),
            "is_starter": side["is_starter"].to_numpy(),
            "fga_per_min": side_profile(side)["fga_per_min"].to_numpy(),
            "fga": drawn["FGA"].mean(axis=0),
            "pts": drawn["PTS"].mean(axis=0),
            "tov": drawn["TOV"].mean(axis=0),
            "pts_p10": np.percentile(drawn["PTS"], 10, axis=0),
            "pts_p90": np.percentile(drawn["PTS"], 90, axis=0),
        }
    )
    return summary.sort_values(["is_starter", "pts"], ascending=[False, False])


shown_shots = pd.concat(
    [
        shot_table(team_abbr, team_role, shots[team_abbr]),
        shot_table(opp_abbr, opp_role, shots[opp_abbr]),
    ],
    ignore_index=True,
)
attempt_means = {
    team_abbr: round(float(shots[team_abbr]["team_fga"].mean()), 1),
    opp_abbr: round(float(shots[opp_abbr]["team_fga"].mean()), 1),
}
print(
    f"live shots for {slate_day.date()}  mean team attempts {attempt_means}"
)
display(shown_shots.round(2))


shot profile fit on 179,467 player-games through 2026-11-04  (each window stops before that game)
live shots for 2026-11-06  mean team attempts {'LAL': 81.7, 'POR': 89.1}


,team,player_name,is_starter,fga_per_min,fga,pts,tov,pts_p10,pts_p90
0,LAL,Luka Dončić,1.0,0.64,25.28,36.89,5.67,24.0,51.0
1,LAL,Austin Reaves,1.0,0.40,13.36,20.96,4.20,12.0,31.0
2,LAL,Walker Kessler,1.0,0.27,10.61,16.38,1.95,8.0,25.0
3,LAL,Ziaire Williams,1.0,0.36,10.24,13.39,1.09,6.0,21.0
4,LAL,Matisse Thybulle,1.0,0.28,7.91,9.64,1.11,3.0,17.0
5,LAL,Quentin Grimes,0.0,0.23,6.68,9.44,1.53,3.0,16.0
6,LAL,Jake LaRavia,0.0,0.16,3.80,5.52,1.32,2.0,11.0
7,LAL,Sandro Mamukelashvili,0.0,0.29,2.27,3.01,0.23,0.0,7.0
8,LAL,Collin Sexton,0.0,0.48,1.19,1.39,0.21,0.0,6.0
9,LAL,Jarred Vanderbilt,0.0,0.15,0.37,0.52,0.19,0.0,2.0


In [7]:
trained_events = training_events(history)
event_model = fit_events(trained_events)
event_profile = live_events(history, players.drop(columns=["player_name", "team"]))
event_profile = names.merge(
    event_profile, on="player_id", how="left", validate="one_to_one"
)
print(
    f"event profile fit on {len(trained_events):,} player-games"
    f" through {history['game_date'].max().date()}"
    "  (each window stops before that game)"
)


def event_profile_for(side):
    ordered = (
        event_profile.set_index("player_id").loc[side["player_id"]].reset_index()
    )
    if len(ordered) != len(side) or not ordered["player_id"].equals(side["player_id"]):
        raise AssertionError("event profile does not match the rotation")
    return ordered


events = allocate_events(
    event_model,
    {
        team_abbr: event_profile_for(team_role),
        opp_abbr: event_profile_for(opp_role),
    },
    shots,
    minutes,
    rng,
)


def event_table(label, side, drawn, other):
    misses = shots[label]["FGA"].sum(axis=1) - shots[label]["FGM"].sum(axis=1)
    other_misses = other["FGA"].sum(axis=1) - other["FGM"].sum(axis=1)
    if np.any(drawn["AST"].sum(axis=1) > shots[label]["FGM"].sum(axis=1)):
        raise AssertionError(f"{label} assists exceed made field goals")
    if np.any(drawn["OREB"].sum(axis=1) + other_rebounds(label) > misses):
        raise AssertionError(f"{label} rebounds exceed its misses")
    if np.any(drawn["STL"].sum(axis=1) > other["TOV"].sum(axis=1)):
        raise AssertionError(f"{label} steals exceed the opponent's turnovers")
    if np.any(drawn["BLK"].sum(axis=1) > other_misses):
        raise AssertionError(f"{label} blocks exceed the opponent's misses")
    if not np.array_equal(drawn["REB"], drawn["OREB"] + drawn["DREB"]):
        raise AssertionError(f"{label} rebounds are not offensive plus defensive")
    summary = pd.DataFrame(
        {
            "team": label,
            "player_name": side["player_name"].to_numpy(),
            "is_starter": side["is_starter"].to_numpy(),
            "ast": drawn["AST"].mean(axis=0),
            "reb": drawn["REB"].mean(axis=0),
            "stl": drawn["STL"].mean(axis=0),
            "blk": drawn["BLK"].mean(axis=0),
        }
    )
    return summary.sort_values(["is_starter", "reb"], ascending=[False, False])


def other_rebounds(label):
    other_label = opp_abbr if label == team_abbr else team_abbr
    return events[other_label]["DREB"].sum(axis=1)


shown_events = pd.concat(
    [
        event_table(team_abbr, team_role, events[team_abbr], shots[opp_abbr]),
        event_table(opp_abbr, opp_role, events[opp_abbr], shots[team_abbr]),
    ],
    ignore_index=True,
)
print(f"live events for {slate_day.date()}")
display(shown_events.round(2))


event profile fit on 179,467 player-games through 2026-11-04  (each window stops before that game)
live events for 2026-11-06


,team,player_name,is_starter,ast,reb,stl,blk
0,LAL,Walker Kessler,1.0,1.41,15.11,1.31,3.20
1,LAL,Luka Dončić,1.0,7.31,8.73,1.58,0.66
2,LAL,Austin Reaves,1.0,4.28,3.51,0.55,0.41
3,LAL,Ziaire Williams,1.0,2.49,3.10,1.84,0.21
4,LAL,Matisse Thybulle,1.0,1.62,2.29,1.30,1.11
5,LAL,Jake LaRavia,0.0,1.94,3.68,1.35,0.89
6,LAL,Quentin Grimes,0.0,3.46,3.56,0.58,0.11
7,LAL,Sandro Mamukelashvili,0.0,0.69,1.63,0.16,0.18
8,LAL,Jarred Vanderbilt,0.0,0.30,0.63,0.09,0.05
9,LAL,Collin Sexton,0.0,0.36,0.29,0.09,0.04


In [9]:
box = assemble_box_score(
    sim,
    {team_abbr: team_role, opp_abbr: opp_role},
    minutes,
    shots,
    events,
)
for label in (team_abbr, opp_abbr):
    same_points = np.array_equal(box.teams[label].stats["PTS"], shots[label]["PTS"])
    same_rebounds = np.array_equal(box.teams[label].stats["REB"], events[label]["REB"])
    same_minutes = np.array_equal(box.teams[label].stats["MIN"], minutes[label])
    if not (same_points and same_rebounds and same_minutes):
        raise AssertionError(f"{label} box score is not the same simulation row")


def box_table(label, side):
    team = box.teams[label]
    summary = pd.DataFrame(
        {
            "team": label,
            "player_name": side["player_name"].to_numpy(),
            "is_starter": side["is_starter"].to_numpy(),
            "min": team.stats["MIN"].mean(axis=0),
            "pts": team.stats["PTS"].mean(axis=0),
            "reb": team.stats["REB"].mean(axis=0),
            "ast": team.stats["AST"].mean(axis=0),
            "stl": team.stats["STL"].mean(axis=0),
            "blk": team.stats["BLK"].mean(axis=0),
            "fgm": team.stats["FGM"].mean(axis=0),
            "fga": team.stats["FGA"].mean(axis=0),
            "fg3m": team.stats["3PM"].mean(axis=0),
            "fg3a": team.stats["3PA"].mean(axis=0),
            "ftm": team.stats["FTM"].mean(axis=0),
            "fta": team.stats["FTA"].mean(axis=0),
            "pts_p10": np.percentile(team.stats["PTS"], 10, axis=0),
            "pts_p90": np.percentile(team.stats["PTS"], 90, axis=0),
        }
    )
    return summary.sort_values(["is_starter", "pts"], ascending=[False, False])


shown_box = pd.concat(
    [
        box_table(team_abbr, team_role),
        box_table(opp_abbr, opp_role),
    ],
    ignore_index=True,
)


def points_over_mean(label, side):
    team = box.teams[label]
    index = int(np.argmax(team.stats["MIN"].mean(axis=0)))
    mean_pts = float(team.stats["PTS"][:, index].mean())
    line = mean_pts - 0.5
    return Prop(label, team.player_ids[index], "PTS", line), side["player_name"].iloc[index]


home_prop, home_name = points_over_mean(team_abbr, team_role)
away_prop, away_name = points_over_mean(opp_abbr, opp_role)
priced = prop_probability(box, [home_prop, away_prop])
print(f"box score for {slate_day.date()}  {box.n_sims:,} simulations, one row each")
print(
    f"P({home_name} over {home_prop.line:.1f} and {away_name} over {away_prop.line:.1f})"
    f"  {priced.joint:.1%}"
    f"  product of the separate chances {priced.independent_product:.1%}"
)
display(shown_box.round(1))


box score for 2026-11-06  5,000 simulations, one row each
P(Luka Dončić over 36.4 and Damian Lillard over 23.1)  23.9%  product of the separate chances 23.2%


,team,player_name,is_starter,min,pts,reb,ast,stl,blk,fgm,fga,fg3m,fg3a,ftm,fta,pts_p10,pts_p90
0,LAL,Luka Dončić,1.0,40.5,36.9,8.7,7.3,1.6,0.7,10.9,25.3,4.8,14.8,10.3,11.6,24.0,51.0
1,LAL,Austin Reaves,1.0,34.5,21.0,3.5,4.3,0.5,0.4,6.8,13.4,1.8,5.0,5.6,6.4,12.0,31.0
2,LAL,Walker Kessler,1.0,40.5,16.4,15.1,1.4,1.3,3.2,7.1,10.6,0.0,0.0,2.2,3.9,8.0,25.0
3,LAL,Ziaire Williams,1.0,29.1,13.4,3.1,2.5,1.8,0.2,4.6,10.2,1.7,4.7,2.4,3.2,6.0,21.0
4,LAL,Matisse Thybulle,1.0,29.1,9.6,2.3,1.6,1.3,1.1,3.4,7.9,2.2,6.1,0.7,0.9,3.0,17.0
5,LAL,Quentin Grimes,0.0,30.0,9.4,3.6,3.5,0.6,0.1,2.9,6.7,1.0,4.1,2.7,3.7,3.0,16.0
6,LAL,Jake LaRavia,0.0,24.6,5.5,3.7,1.9,1.4,0.9,2.2,3.8,0.8,1.6,0.2,0.5,2.0,11.0
7,LAL,Sandro Mamukelashvili,0.0,7.8,3.0,1.6,0.7,0.2,0.2,1.1,2.3,0.3,1.1,0.6,0.7,0.0,7.0
8,LAL,Collin Sexton,0.0,2.4,1.4,0.3,0.4,0.1,0.0,0.5,1.2,0.1,0.4,0.2,0.3,0.0,6.0
9,LAL,Jarred Vanderbilt,0.0,2.4,0.5,0.6,0.3,0.1,0.1,0.2,0.4,0.1,0.1,0.0,0.1,0.0,2.0
